# **MC2融合算子概念介绍**
本章将介绍MC2融合算子的基础理论，内容涵盖基本概念、MC2 MTE通信算子、MTE通信机制、PUT/GET通信方向、Win Area内存布局、集合通信算法与核心优势。我们将按照以下结构，带你逐步理解MC2融合算子：
- 什么是MC2融合算子
- MC2 MTE通信算子介绍
- MTE Win Area通信机制
- PUT/GET两种通信方向
- Win Area内存布局
- 集合通信算法
- MC2融合算子的优势
- 课后练习
---

## **1. 什么是MC2融合算子**
MC2（Matrix Computation & Communication）融合算子是Ascend平台上**通信与计算融合**的算子范式，通过将计算操作与通信操作进行融合，实现计算通信重叠，有效隐藏通信延迟，提升分布式训练的整体性能。

在多卡分布式训练场景下，设备间需要频繁进行数据通信（如AllReduce、AllGather、ReduceScatter等集合通信操作），通信开销往往成为性能瓶颈。MC2融合算子通过精细的任务调度，将计算密集型操作与通信密集型操作进行重叠，使得在等待数据传输的同时仍能进行有效计算，从而最大化硬件资源利用率。

### **两种通信方式的区别**

| 维度 | HCCL 高阶 API | MTE 直接通信（本教程） |
|------|--------------|------------------------|
| 通信控制 | `Hccl` 对象封装，调用 `AllGather`/`Wait` | MTE 引擎直接读写 Win Area |
| Kernel 启动 | 不支持 `<<<>>>` 直调 | 支持 `<<<>>>` 直调 |
| 编译方式 | bisheng (.asc) | CMake ascendc (.cpp) |
| 同步机制 | `Commit`/`Wait` 事件驱动 | 状态区轮询（自定义同步） |
| 适用场景 | 标准通算融合 | 需要精细控制通信流程的场景 |
| 平台 | A2/A3/950 | **仅 Ascend 950（arch35）** |

---

## **2. MC2 MTE通信算子介绍**
MC2 MTE通信算子是使用**MTE引擎直接读写Win Area**的Kernel直调算子，通过精细控制通信流程，实现计算与通信的高效融合。

### **核心特点**

**1. MTE引擎直调通信**
- 使用 `SetCommEngine(3)` 设置MTE引擎直接读写Win Area
- 不依赖HCCL服务端调度，通信过程完全由Kernel控制
- 支持自定义分块、错序访问、多split处理等精细控制

**2. Kernel直调模式**
- 支持 `<<<>>>` Kernel直调，通信与计算在同一Kernel中执行
- 编译方式为CMake + ascendc (.cpp)，不支持bisheng
- Kernel参数包含 `mc2Context`（HcclA5OpResParam结构体）

**3. Win Area通信机制**
- 每个rank拥有一块Win Area共享内存，对所有rank可见
- PUT模式：本卡主动写入所有rank的Win Area
- GET模式：本卡写入本地Win Area，其他rank主动读取
- 状态区轮询实现同步，无需HCCL事件驱动

**4. 支持的集合通信**

| 集合通信 | PUT模式支持 | GET模式支持 | 典型算子 |
|---------|-------------|-------------|----------|
| **AllGather** | ✓ | ✓ | AllGatherMatmul (PUT), Abs+AllGather (GET) |
| **AllToAll** | ✓ | ✓ | AllToAllMatmul |
| **ReduceScatter** | ✓ | ✓ | MatmulReduceScatter (GET), ReduceScatter+Abs (PUT) |
| **AllReduce (one-shot)** | ✓ | ✓ | MatmulAllReduce (小数据) |
| **AllReduce (two-shot)** | ✓ | ✓ | MatmulAllReduce (大数据) |

### **典型应用场景**

**场景1：Matmul + ReduceScatter（GET模式）**
- AIC执行Matmul，结果直接写入本卡Win Area
- AIV从所有rank Win Area读取本卡chunk分区，做ReduceSum
- 输出大小减少rankDim倍
- **优化点：省掉GM→Win搬运**

**场景2：AllGather + Matmul（PUT模式）**
- 本卡数据先写入所有rank的Win Area（先通信）
- 各rank从本卡Win Area读取数据执行Matmul（后计算）
- 适合通信数据是原始输入的场景

**场景3：大数据量AllReduce（two-shot）**
- 第一次通信：ReduceScatter，输出减少rankDim倍
- 第二次通信：AllGather，恢复原始大小
- 通信量从 `(N-1)×size` 降低到 `2×size`
- **注意：两次通信间必须清状态区**

### **开发流程**

```
MC2 MTE通信算子开发流程:

Step 1: 环境检查
  └── Ascend 950平台，多卡环境
  └── HCCL_BUFFSIZE设置（Win Area大小）

Step 2: 选择通信方向
  └── 通信数据是原始输入 → PUT模式
  └── 通信数据是计算结果 → GET模式

Step 3: 选择集合通信算法
  └── AllGather / AllToAll / ReduceScatter
  └── AllReduce: 小数据用one-shot，大数据用two-shot

Step 4: 实现Kernel
  └── 初始化mc2Context（HcclA5OpResParam）
  └── 实现通信流程（PUT: PutDataToAllWin / GET: ComputeToWin）
  └── 实现状态同步（WriteStatusToWin + ReadStatus）
  └── 实现数据汇聚（GatherFromWin / ReduceGatherFromWin）

Step 5: 编译运行
  └── CMake + ascendc编译
  └── HcclCommInitAll初始化通信域
  └── <<<>>> Kernel直调启动
```

### **Host侧关键API**

| API | 作用 |
|-----|------|
| `HcclCommInitAll` | 单进程多线程初始化通信域（无需RANK_TABLE_FILE） |
| `Mc2CcTilingConfig` | 配置MC2 Tiling参数 |
| `SetCommEngine(3)` | 设置MTE引擎直接通信 |
| `HcclAllocComResourceByTiling` | 分配Win Area资源，填充mc2Context |
| `<<<>>>` | Kernel直调启动 |

---

## **3. MTE Win Area通信机制**
MTE（Memory Transfer Engine）是Ascend NPU上的数据搬运引擎。在MC2算子中，MTE被用于在rank间通过**共享内存窗口（Win Area）**直接传输数据，无需经过HCCL服务端调度。

**核心思路：** 每个rank拥有一块Win Area内存，该内存对所有rank可见。rank间通过MTE引擎直接读写彼此的Win Area，配合状态标志实现同步。

### **状态同步机制**

每个AIV核向**所有rank**（包括自己）的状态区写入"已就绪"标志（值为1.0）。接收方轮询读取自己rank的状态区，检查所有rank是否都已写入标志。

**验证方法：**
1. 读取当前核对应的状态区段（`rankDim * FLOAT_ALIGN_NUM` 个float）
2. `Sum` 求和得到总值
3. 判断：`sum ∈ [rankDim - 0.5, rankDim + 0.5)` → 所有rank就绪
4. 就绪后清零状态区，为下一轮通信做准备

---

## **4. PUT/GET两种通信方向**
MTE Win Area通信按**谁主动搬运数据**分为PUT（写模式）和GET（读模式）两种方向。

### **PUT模式（写模式，先通信后计算）**

**适用场景：** 通信数据就是原始输入，计算依赖通信完成（如 `AllGatherMatmul`、`AllGather+Abs`）。

**语义：** 本卡主动把自己的数据写入**所有rank**（包括自己）的Win Area；写入位置按rank分区。

**流程：** `PutDataToAllWin → WritePutStatus → ReadPutStatus → [计算] + GatherFromSelfWin`

```
PUT 执行流程：
┌─────────────────────────────────────────────────────────┐
│  1. PutDataToAllWin:  本卡数据 → 所有 rank 的 Win Area   │
│  2. WritePutStatus:   通知所有 rank 本卡已写入          │
│  3. ReadPutStatus:    等待所有 rank 写入本卡 Win Area   │
│  4. GatherFromSelfWin: 本卡 Win Area → GM(输出)         │
│     [可选：在 GatherFromSelfWin 前或后插入计算]         │
└─────────────────────────────────────────────────────────┘
```

### **GET模式（读模式，先计算后通信）**

**适用场景：** 通信内容依赖于本卡计算结果，必须先算完再让别人读取（如 `MatmulReduceScatter`、`Abs+AllGather`）。

**语义：** 本卡只把"计算结果"写入自己的Win Area，其他rank主动来读取。

**流程：** `ComputeToWin → WriteStatusToWin → ReadStatus → GatherFromWin`

```
GET 执行流程：
┌─────────────────────────────────────────────────────────┐
│  1. ComputeToWin:    GM(输入) → UB → 计算 → 本地 Win Area│
│  2. WriteStatusToWin: 通知所有 rank 本卡数据就绪        │
│  3. ReadStatus:      等待所有 rank 数据就绪             │
│  4. GatherFromWin:   远端 Win Area → UB → GM(输出)      │
└─────────────────────────────────────────────────────────┘
```

### **PUT vs GET 选择原则**

| 判断条件 | 选择 |
|---------|------|
| 通信内容是原始输入，计算可在通信完成后做 | PUT（先通信后计算） |
| 通信内容依赖本卡计算结果 | GET（先计算后通信） |
| 典型算子 AllGatherMatmul / AllGather+Abs | PUT |
| 典型算子 MatmulReduceScatter / Abs+AllGather | GET |

### **GET模式的优化优势**

GET模式将计算结果由 `ComputeToWin` 一次性写入本地Win Area，**省掉了先 `CopyDataToWin`（GM→UB→Win）再读回计算的整次搬运**。

---
## **5. Win Area内存布局**
每个rank的Win Area由 `HcclAllocComResourceByTiling` 分配，分为两个分区（双缓冲），通过 `winBufferFlags` 交替使用。

### **Win Area布局结构**

```
windowsIn[rankId] 指向的内存空间：
┌──────────────────────────────────────────────────────┐
│ 偏移 0                                               │
│ ┌──────────────┐                                     │
│ │ 状态区 0      │  [0, 462KB)        单状态区大小     │
│ ├──────────────┤                                     │
│ │ 状态区 1      │  [462KB, 924KB)    双缓冲状态区     │
│ ├──────────────┤                                     │
│ │ 标志区        │  [924KB, 1MB]      Win分区标志      │
│ ├──────────────┤                                     │
│ │ 数据区 0      │  [1MB, 1MB+dataSize)  winFlag==0   │
│ └──────────────┘                                     │
└──────────────────────────────────────────────────────┘

windowsOut[rankId] 指向的独立内存空间：
┌──────────────────────────────────────────────────────┐
│ 数据区 1        │  [0, dataSize)      winFlag==1     │
└──────────────────────────────────────────────────────┘
```

### **关键常量**

| 常量 | 值 | 说明 |
|------|-----|------|
| `STATE_WIN_SIZE` | 1MB | 状态+标志区总大小 |
| `SINGLE_STATE_REGION_SIZE` | 462KB | 单个状态分区大小 |
| `WIN_ADDR_ALIGN` | 512B | Win地址对齐要求 |
| `FLOAT_ALIGN_NUM` | 8 | 状态值对齐元素数 |

### **PUT模式下Win Area按rank分区布局**

```
本卡 Win Area (PUT模式):
┌────────────────────────────────────────────────────┐
│ 状态区                │ [0, 1MB)                  │
├────────────────────────────────────────────────────┤
│ rank0 数据区          │ [1MB, 1MB + perRankLen)   │ ← rank0 写入
│ rank1 数据区          │ [...]                     │ ← rank1 写入
│ rank2 数据区          │ [...]                     │ ← rank2 写入
│ rank3 数据区          │ [...]                     │ ← rank3 写入
└────────────────────────────────────────────────────┘

容量：STATE_WIN_SIZE + rankDim * perRankDataSize
```

---
## **6. 集合通信算法**
MC2算子支持多种集合通信算法，可根据数据规模选择最优实现。

### **集合通信一览表**

| 集合通信 | 实现路径 | Reduce 计算 | 通信量 | 适用数据规模 |
|---------|---------|-----------|-------|-----------|
| **AllGather** | 模板原生支持 | 无 | `N × size` | —— |
| **AllToAll** | 按 chunk 分段，第 i 段发给 rank i | 无 | `N × size` | —— |
| **ReduceScatter** | AllToAll + 本卡 ReduceSum | AIV `Add` | `N × size` | —— |
| **AllReduce (one-shot)** | AllGather + 本卡 ReduceSum | AIV `Add` | `(N-1) × size` | **小数据** |
| **AllReduce (two-shot)** | ReduceScatter + AllGather | AIV `Add` | `2 × size` | **大数据** |

（`N` = rankDim，`size` = 单卡原始数据量）

### **AllReduce两种算法选择**

**One-shot AllReduce = AllGather + 本卡 ReduceSum**
- 流程：先做一次AllGather，再在本卡做Reduce
- 通信量：`(N-1) × size`，单次通信，流程短
- 适用：**小数据**场景

**Two-shot AllReduce = ReduceScatter + AllGather**
- 流程：先ReduceScatter得到本卡chunk，再AllGather广播
- 通信量：`2 × size`，流程长但通信量低
- 适用：**大数据**场景
- **注意：两次通信之间必须清状态区！**

| 条件 | 选择 |
|------|------|
| 数据量小（典型 < 数 MB/卡） | **one-shot** |
| 数据量大（典型 ≥ 数 MB/卡） | **two-shot** |
| 卡数少（N ≤ 2–4） | one-shot |
| 卡数多（N ≥ 8） | two-shot 优势放大 |

In [ ]:
import numpy as np

# ===================== 集合通信算法模拟演示 =====================
RANK_DIM = 4              # rank数量
PER_RANK_DATA_MB = 10     # 单卡数据量(MB)

def simulate_allgather():
    """AllGather: 每卡收齐所有rank的完整数据"""
    comm_volume = RANK_DIM * PER_RANK_DATA_MB  # N × size
    output_size = RANK_DIM * PER_RANK_DATA_MB  # 输出为完整数据
    return comm_volume, output_size, "每卡持有所有rank数据"

def simulate_allreduce_one_shot():
    """One-shot AllReduce: AllGather + ReduceSum"""
    comm_volume = (RANK_DIM - 1) * PER_RANK_DATA_MB  # (N-1) × size
    output_size = PER_RANK_DATA_MB  # 输出为求和结果
    return comm_volume, output_size, "流程短，适合小数据"

def simulate_allreduce_two_shot():
    """Two-shot AllReduce: ReduceScatter + AllGather"""
    comm_volume = 2 * PER_RANK_DATA_MB  # 2 × size
    output_size = PER_RANK_DATA_MB  # 输出为求和结果
    return comm_volume, output_size, "流程长但通信量低，适合大数据"

print("=== 集合通信算法对比演示 ===")
print(f"假设: {RANK_DIM}个rank, 单卡数据量 {PER_RANK_DATA_MB}MB\n")

for name, func in [("AllGather", simulate_allgather), 
                  ("AllReduce(one-shot)\", simulate_allreduce_one_shot),
                  ("AllReduce(two-shot)\", simulate_allreduce_two_shot)]:
    comm_vol, out_size, desc = func()
    print(f"{name}:")
    print(f"  通信量: {comm_vol}MB")
    print(f"  输出大小: {out_size}MB")
    print(f"  说明: {desc}\n")

# 选择建议演示
print("=== 算法选择建议 ===")
if PER_RANK_DATA_MB < 5:
    print(f"数据量 {PER_RANK_DATA_MB}MB < 5MB，建议使用 one-shot AllReduce")
else:
    print(f"数据量 {PER_RANK_DATA_MB}MB >= 5MB，建议使用 two-shot AllReduce")
    print(f"  one-shot通信量: {(RANK_DIM-1) * PER_RANK_DATA_MB}MB")
    print(f"  two-shot通信量: {2 * PER_RANK_DATA_MB}MB")
    print(f"  通信量节省: {((RANK_DIM-1) - 2) * PER_RANK_DATA_MB}MB ({((RANK_DIM-3)/(RANK_DIM-1)*100):.1f}%)")

---
## **7. MC2融合算子的优势**
MC2融合算子是分布式训练性能优化的关键技术，通过计算通信重叠实现性能提升，主要优势如下：  

- **隐藏通信延迟：** 传统模式下通信期间计算资源空闲，MC2融合通过计算通信重叠，将通信延迟隐藏在计算过程中，有效降低整体耗时。  

- **提升资源利用率：** 计算单元与通信单元并行工作，最大化硬件资源利用率，避免资源闲置。  

- **减少数据搬运：** GET模式将计算结果直接写入Win Area，省掉中间搬运步骤，减少GM→UB→Win的整次数据传输。  

- **精细控制通信流程：** 通过MTE引擎直接读写Win Area，可实现自定义分块、错序访问、与任意AIV计算融合等精细控制。  

- **灵活的集合通信支持：** 支持AllGather、AllToAll、ReduceScatter、AllReduce等多种集合通信算法，可按数据规模选择最优实现。  

---

# **课后练习**
请根据本章内容认真思考，选出正确选项。  

1. （单选题）MC2 MTE通信算子的核心特点是？  
    A. 使用HCCL高阶API进行通信调度。    
    B. 使用MTE引擎直接读写Win Area，支持Kernel直调。     
    C. 使用bisheng编译方式。    
    D. 支持A2/A3/950所有平台。  


2. （单选题）MC2 MTE通信算子支持Kernel直调的原因是？  
    A. 使用HCCL的Commit/Wait机制。  
    B. 通信与计算在同一Kernel中执行，使用<<<>>>启动。   
    C. 需要通过服务端调度。  
    D. 仅支持单卡执行。    

3. （单选题）MC2 MTE通信算子支持的集合通信类型包括？  
    A. 仅支持AllGather。  
    B. 仅支持AllReduce。   
    C. 支持AllGather、AllToAll、ReduceScatter、AllReduce。  
    D. 仅支持单卡通信。    

4. （单选题）关于MC2 MTE通信算子开发流程，以下说法正确的是？    
    A. 使用HcclCommInitAll需要配置RANK_TABLE_FILE。  
    B. SetCommEngine(3)用于设置MTE引擎直接通信。   
    C. 编译方式为bisheng。  
    D. 仅支持单进程单线程模式。    

5. （多选题）关于MC2 MTE通信算子的典型应用场景，以下说法正确的是？    
    A. Matmul+ReduceScatter使用GET模式，省掉GM→Win搬运。   
    B. AllGather+Matmul使用PUT模式，适合先通信后计算。   
    C. 大数据AllReduce使用two-shot算法降低通信量。  
    D. 仅支持小数据量场景。    

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/01.02_answer.txt